## Inspect SOPE Gym `pdataset`

This notebook inspects the SOPE Gym adapter under `src/sope_interface` and shows the chunk fields passed into `SopeDiffuser`.


In [ ]:
import sys
from pathlib import Path
from flax import nnx

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.sope_interface.dataset import (
    SopeGymChunkDatasetConfig,
    load_sope_gym_dataset,
    make_sope_gym_chunk_dataloader,
    split_sope_gym_episodes,
    summarize_sope_gym_episodes,
)

data_dir = REPO_ROOT / "data" / "sope_gym_data" / "pdataset"
bundle = load_sope_gym_dataset(data_dir)
episodes = split_sope_gym_episodes(bundle)
cfg_dataset = SopeGymChunkDatasetConfig(
    chunk_size=8,
    stride=1,
    frame_stack=1,
    state_dim=bundle.observations.shape[-1],
    action_dim=bundle.actions.shape[-1],
    normalize=True,
    normalization_source="asset",
    return_metadata=True,
)

loader, stats = make_sope_gym_chunk_dataloader(
    episodes=episodes,
    config=cfg_dataset,
    batch_size=4,
    num_workers=0,
    shuffle=False,
    drop_last=False,
)
first_batch = next(iter(loader))

nnx.display(
    {
        "data_dir": str(data_dir),
        "cfg_dataset": cfg_dataset,
        "bundle_shapes": {
            "observations": bundle.observations.shape,
            "actions": bundle.actions.shape,
            "rewards": bundle.rewards.shape,
            "terminals": bundle.terminals.shape,
        },
        "reference_normalization": bundle.normalization,
        "episode_summary": summarize_sope_gym_episodes(episodes, max_episodes=3),
        "dataset": loader.dataset,
        "normalization_stats": stats,
        "first_batch": first_batch,
    }
)


In [ ]:
item0 = loader.dataset[0]
nnx.display(item0)
